In [ ]:
# Weekly Homework 9: Restaurant Reviews

### Author:

## Introduction to Machine Learning

#### University of Redlands - DATA 301
#### Prof: Joanna Bieri [joanna_bieri@redlands.edu](mailto:joanna_bieri@redlands.edu)
#### [Class Website](https://joannabieri.com/machine_learning.html)

---

**Due Sunday 11/15 at 11:59pm.** This covers embeddings, RNNs and GRUs, and attention.

"The pasta was not terrible" is a good review! The word "terrible" is in it, but the word "not" right before it flips the meaning. This week we find out which networks can learn that. You build a network that ignores the order of the words, a GRU that reads them in order, and a GRU with **attention**, and then you look inside the attention model to see which words it pays attention to.

GOALS:

1. Look at text data and work out the rule behind the labels.
2. Use a baseline network that ignores word order, then write your own networks as classes (`nn.Module`).
3. Train a GRU and a GRU with attention, and look at where the attention goes.
4. Use the test set once and explain what the model can and can not do.

**What you can copy.** Copy freely from the RNN and attention notes and the practice problems, and change the names. The `ReviewNet` class and the "What this looks like in real life" cell at the end of the RNN notes are your best guide. If you want to write or use helper functions that is okay too, just comment each line so it is clear you understand what is going on. Every sentence you write is yours.

**Make your guess first.** Question 1b asks for a guess before you train anything. Write it down and do not change it later, even if it turns out wrong!

**The data.** `data/restaurant_reviews.csv`. These are 6,000 made up restaurant reviews. Every review is 40 words long, made of 10 short pieces. One piece is the **key phrase**, like "pasta was so amazing" or "waiter was not great". The label is 1 for a positive review and 0 for a negative review, and it comes from the key phrase: a positive word ("great", "good", "amazing", "excellent", "wonderful") gives a positive review, a negative word ("bad", "awful", "terrible", "horrible", "disappointing") gives a negative review, and "not" right before the feeling word flips it. The other 9 pieces do not matter ("we came for dinner", "it was a friday"), and some of them have "not" or "so" in them too ("it was not raining"). Just like real reviews, some people clicked the wrong star rating: **about 1 review in 25 has the wrong label.** The reviews do not read like real sentences, and that is on purpose, because we know exactly which words matter. Claude (an AI assistant from Anthropic) wrote the code that makes these reviews for me, and I checked it.

All 6,000 reviews come from the same pool, in random order. The first 4,000 are for training, the next 1,000 are for validation, and the last 1,000 are our test set.

**How long does it take?** Plan on about two hours. All of the code together runs in a minute or two on a laptop.

**How to turn this in.** Your repository on GitHub **is** your submission. There is nothing to hand in on Canvas.

1. The finished work is on your **`main`** branch.
2. It is **pushed to GitHub** before the deadline.
3. Your name is on the **Author** line at the top of this notebook.

```bash
git add .
git commit -m "Weekly homework 9"
git push
```

I grade from whatever is on GitHub at the deadline. If it is not pushed, I cannot see it.

---

# Part 1: Read the reviews

The code in this part is given. Run it and look at the output.

**1a.** How many positive and negative reviews are there? How many key phrases use "not", and how many reviews have the word "not" somewhere in them? Why can a model not just count the word "not" to get the answer?

**1b.** Read the six reviews that are printed. Each review is split into its 10 pieces, and the key phrase is in CAPITALS. For each one, use the rule to say what the label should be. One of the six has the wrong label (a wrong star). Which one? Then make a guess: will a network that **ignores the order of the words** do well on this data? Why or why not? Write your guess down now and do not change it later!

In [ ]:
# RUN THIS. Imports, load the reviews, and find the key phrase in each one.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

df = pd.read_csv("data/restaurant_reviews.csv")
reviews = list(df["review"])                     # 6000 reviews, each one a string of 40 words
labels = df["label"].values                      # 1 = positive, 0 = negative

feeling_words = ["great", "good", "amazing", "excellent", "wonderful",
                 "bad", "awful", "terrible", "horrible", "disappointing"]

def key_phrase(review):
    words = review.split()                       # the review as a list of words
    for j in range(len(words)):
        if words[j] in feeling_words:            # the feeling word is the last word of the key phrase
            return " ".join(words[j - 3:j + 1])

uses_not = np.array([key_phrase(r).split()[2] == "not" for r in reviews])   # True if the key phrase has "not"

print("reviews:", len(reviews))
print("positive:", (labels == 1).sum(), "  negative:", (labels == 0).sum())
print("key phrases with 'not':", uses_not.sum())
print("reviews with 'not' somewhere:", sum(["not" in r.split() for r in reviews]))
print()
import textwrap
def show_review(review, label):
    words = review.split()
    phrases = [" ".join(words[k:k + 4]) for k in range(0, len(words), 4)]   # the 10 pieces of 4 words
    phrases = [p.upper() if p == key_phrase(review) else p for p in phrases]   # the key phrase in CAPITALS
    phrases = [p.replace(" ", "\u00a0") for p in phrases]                 # so a line never breaks inside a piece
    print("label:", label, "(positive)" if label == 1 else "(negative)")
    print(textwrap.fill(" | ".join(phrases), width=90, initial_indent="    ", subsequent_indent="    "))
    print()

for i in range(6):
    print("review", i)
    show_review(reviews[i], labels[i])

*Your answers here.*

---

# Part 2: Words to numbers, and a model that ignores order

A network needs numbers, so every word gets an id, like in the RNN notes. We build the list of words, the **vocabulary**, from the training reviews only. The split is given, and the test reviews stay alone until Part 4.

How the vocabulary is made, in the next cell: `" ".join(reviews[:4000]).split()` puts all of the training reviews into one long list of words. `set(...)` keeps just one copy of every word, and `sorted(...)` puts them in alphabetical order, so everyone gets the same list. Then `word_to_id` is a dictionary that gives every word its spot in that list, eg. `word_to_id["pasta"]` is the id of "pasta". Finally every review is turned into a list of 40 ids.

The training function is the one from the RNN notes, and there is one more helper that prints the accuracy separately for the reviews whose key phrase has "so" and the ones whose key phrase has "not". Run both cells.

Our **baseline** does not read in order at all. It looks up the embedding of every word and takes the **average** of the 40 embeddings, then one `nn.Linear` layer turns that average into 2 scores. Shuffle the words of a review and this model gives exactly the same answer. Its class is given: read the comments, then run it.

**2a.** What is the validation accuracy of the averaging model after 10 epochs? Look at the two accuracies from `accuracy_by_key`. Which reviews does it get right, and which does it get wrong? Explain why. (Hint: "pasta was not terrible ... it was so busy" and "pasta was so terrible ... it was not busy" use exactly the same words.)

In [ ]:
# RUN THIS. Every word gets an id, then the split.
vocab = sorted(set(" ".join(reviews[:4000]).split()))     # every word that shows up in the training reviews
vocab_size = len(vocab)
word_to_id = {}
for i in range(vocab_size):
    word_to_id[vocab[i]] = i                              # word_to_id["pasta"] is the id of "pasta"

X = torch.tensor([[word_to_id[w] for w in r.split()] for r in reviews])   # (6000 reviews, 40 words)
y = torch.tensor(labels)

X_train, y_train = X[:4000], y[:4000]
X_valid, y_valid = X[4000:5000], y[4000:5000]
uses_not_valid = uses_not[4000:5000]

print("vocabulary size:", vocab_size)
print("X_train:", X_train.shape, "  X_valid:", X_valid.shape)
print("the first review as ids:", X_train[0])

In [ ]:
# RUN THIS. The training function from the RNN notes, and accuracy for "so" and "not" reviews.
def train_model(model, X_train, y_train, X_valid, y_valid, n_epochs):
    torch.manual_seed(42)                         # so the batches come in the same order every time
    train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=32, shuffle=True)
    loss_fn = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

    valid_accuracy = []
    for epoch in range(n_epochs):
        model.train()
        for X_batch, y_batch in train_loader:
            optimizer.zero_grad()
            scores = model(X_batch)
            loss = loss_fn(scores, y_batch)
            loss.backward()
            optimizer.step()

        model.eval()
        with torch.no_grad():
            y_pred = model(X_valid).argmax(dim=1)
            accuracy = (y_pred == y_valid).float().mean().item()
        valid_accuracy.append(accuracy)
        print("epoch", epoch + 1, "  valid accuracy:", round(accuracy, 4))

    return valid_accuracy

def accuracy_by_key(model, X, y, uses_not):
    model.eval()
    with torch.no_grad():
        right = (model(X).argmax(dim=1) == y).numpy()          # True where the model is right
    print("key phrase with 'so': ", (~uses_not).sum(), "reviews,  accuracy", round(right[~uses_not].mean(), 4))
    print("key phrase with 'not':", uses_not.sum(), "reviews,  accuracy", round(right[uses_not].mean(), 4))

In [ ]:
# RUN THIS. 2a: the averaging model (given). Read the comments, then run it.

# Step 1: the class
class AverageNet(nn.Module):

    def __init__(self):
        super().__init__()                            # always the first line
        self.embed = nn.Embedding(vocab_size, 8)      # every word becomes 8 numbers
        self.out = nn.Linear(8, 2)                    # 2 scores: negative, positive

    def forward(self, X):
        vectors = self.embed(X)                       # (reviews, 40) -> (reviews, 40, 8)
        average = vectors.mean(dim=1)                 # the average of the 40 words: (reviews, 8)
        return self.out(average)

# Step 2: train it
torch.manual_seed(42)
avg_model = AverageNet()
accuracy_avg = train_model(avg_model, X_train, y_train, X_valid, y_valid, n_epochs=10)

# Step 3: "so" and "not" reviews
accuracy_by_key(avg_model, X_valid, y_valid, uses_not_valid)

*Your answers here.*

---

# Part 3: A GRU, then a GRU with attention

**3a.** Write a GRU class like `ReviewGRU` from the RNN notes, following the steps in the comments: an embedding with 8 numbers per word, a GRU memory that goes from 8 to 32 numbers, and a Linear layer on the **last** hidden state. Train it for 10 epochs and run `accuracy_by_key`. How does it compare to the averaging model, on the "so" reviews and on the "not" reviews? Why can a GRU handle "not" when the averaging model can not?

Next **you** write the attention model, following the steps in the comments. It uses the same steps as the attention notes: dot products, a softmax, and a weighted average. The difference is that there is only ONE query, and the network learns it. First the GRU reads the review. Then, instead of using only the last hidden state, the query is compared to the hidden state after **every** word (all 40 of them, `all_hidden`). The softmax turns the 40 scores into **attention weights** (40 numbers between 0 and 1 that add up to 1), and the summary of the review is the weighted average of the hidden states. So here the hidden states are the keys AND the values. Then train it.

**3b.** Train the attention model for 10 epochs and run `accuracy_by_key`. Then run the two plot cells. Look at the learning curves: which model gets to its best validation accuracy in the fewest epochs? Look at the attention weights for the three reviews: which word gets almost all of the weight?

**3c.** In the third review (review 2) the key phrase has "not", but the attention weight on the word "not" is tiny. How can the model still get it right? (Hint: what has the GRU already read when its hidden state reaches the feeling word?) Then, which model would you keep, the GRU or the GRU with attention? Use the numbers and the plots.

In [ ]:
# 3a: the GRU

# Step 1: write a class ReviewGRU(nn.Module). Start __init__ with super().__init__(), then make three layers:
#   - self.embed: an embedding that turns every word id into 8 numbers (there are vocab_size words)
#   - self.memory: a GRU that takes the 8 numbers for each word and keeps a memory of 32 numbers (use batch_first=True)
#   - self.out: a Linear layer that turns the 32 numbers of the LAST memory into 2 scores
# In forward(self, X):
#   1. embed the words
#   2. run them through the GRU. It gives back TWO things: the memory after every word, and the last memory
#   3. send the last memory through self.out. The last memory has shape (1, reviews, 32), so use [0] to drop the 1

# Step 2: torch.manual_seed(42), gru_model = ReviewGRU(), then
#         accuracy_gru = train_model(...) for 10 epochs

# Step 3: accuracy_by_key for gru_model on the validation reviews

In [ ]:
# 3b: the GRU with attention. It looks back at the hidden state after EVERY word.

# Write a class AttentionGRU(nn.Module). Start __init__ with super().__init__(), then make four layers:
#   - self.embed and self.memory: the same embedding (8 numbers per word) and GRU (8 to 32) as your ReviewGRU
#   - self.query: the learned query, a Linear layer from 32 numbers to 1 score with bias=False.
#     Comparing the query with a hidden state is a dot product, and that is exactly what a Linear layer with no bias does
#   - self.out: a Linear layer that turns the 32 numbers of the summary into 2 scores
# In forward(self, X):
#   1. embed the words and run them through the GRU. This time keep the memory after EVERY word, all_hidden: (reviews, 40, 32)
#   2. send all_hidden through self.query to get one score for every word. It comes out (reviews, 40, 1), so use [:, :, 0] to get (reviews, 40)
#   3. softmax the scores along the words (dim=1), so the 40 attention weights add up to 1 for each review
#   4. save the weights with self.weights = weights (the plot cells below need them!)
#   5. the summary is the weighted average of the 40 hidden states: multiply each hidden state by its weight and add them up
#      over the words. Hint: (weights[:, :, None] * all_hidden).sum(dim=1) gives (reviews, 32)
#   6. send the summary through self.out

In [ ]:
# 3b: train the attention model
# Step 1: torch.manual_seed(42), attention_model = AttentionGRU(),
#         accuracy_attention = train_model(...) for 10 epochs
# Step 2: accuracy_by_key for attention_model on the validation reviews


In [ ]:
# RUN THIS after 2a, 3a and 3b. The three learning curves.
epochs = range(1, 11)
plt.figure(figsize=(7, 4))
plt.plot(epochs, accuracy_avg, marker="o", label="averaging model")
plt.plot(epochs, accuracy_gru, marker="o", label="GRU")
plt.plot(epochs, accuracy_attention, marker="o", label="GRU with attention")
plt.xlabel("epoch")
plt.xticks(epochs)                   # only whole epochs on the x axis
plt.ylabel("validation accuracy")
plt.ylim(0.4, 1.02)
plt.legend()
plt.grid()
plt.show()

In [ ]:
# RUN THIS after 3b. Where does the attention go? The weights for three validation reviews.
show = [0, 1, 2]                                    # pick any three validation reviews
attention_model.eval()
with torch.no_grad():
    attention_model(X_valid[show])                  # this saves attention_model.weights for these reviews

fig, axes = plt.subplots(len(show), 1, figsize=(12, 3 * len(show)))
for ax, i, w in zip(axes, show, attention_model.weights):
    words = reviews[4000 + i].split()               # validation review i is row 4000 + i
    ax.bar(range(40), w.numpy())
    ax.set_xticks(range(40))
    ax.set_xticklabels(words, rotation=90)
    ax.set_ylabel("attention weight")
    ax.set_title("key phrase: " + key_phrase(reviews[4000 + i]) + "   label: " + str(labels[4000 + i]))
plt.tight_layout()
plt.show()

for i, w in zip(show, attention_model.weights):
    words = reviews[4000 + i].split()
    top = w.argmax().item()                         # the word with the biggest weight
    print("review", i, "  biggest weight:", round(w[top].item(), 3), "on", words[top],
          "   weight on the word before it:", round(w[top - 1].item(), 3), "on", words[top - 1])

*Your answers here.*

---

# Part 4: The test set, once

Now the last 1,000 reviews come out. **BEWARE! They get exactly the same steps as the training reviews:** the same `word_to_id` (we do not build a new vocabulary for them). We already made them into ids in Part 2, so here we only take the rows.

**4a.** Compute the test accuracy of the model you chose in 3c, **one time**, and the averaging model's test accuracy. Compare them to their validation accuracies. Neither model gets close to 1.0. Using what you know about the labels, what is the best accuracy any model could get on these 1,000 reviews? Why do we only look at the test set once?

**4b.** The second cell (given) makes a table of every test review the model got wrong: the key phrase, the label, and what the model said. Use the rule from the top of the notebook on a few of them. Who is wrong, the model or the label? What does that tell you about the model?

**4c.** The owner of a restaurant wants to use your model to sort online reviews into good and bad. Write three or four sentences to them. Say what the model does, how often it is wrong, and one reason to be careful with it. No jargon: no "accuracy", no "GRU", no "attention".

In [ ]:
# 4a: the test set, once
# Step 1: X_test, y_test = X[5000:], y[5000:]   and   uses_not_test = uses_not[5000:]
# Step 2: your chosen model in .eval() mode, and inside torch.no_grad():
#         y_pred_test = the model's predictions on X_test (argmax), and test_accuracy against y_test
# Step 3: the averaging model's accuracy on the same test reviews, call it avg_test_accuracy
# Step 4: print the test accuracy, the chosen model's last validation accuracy, and avg_test_accuracy


In [ ]:
# RUN THIS after 4a. 4b: every test review the model got wrong
wrong = np.where(y_pred_test != y_test)[0]
print("wrong:", len(wrong), "of", len(y_test))
print()
names = {0: "negative", 1: "positive"}
table = pd.DataFrame({
    "key phrase": [key_phrase(reviews[5000 + i]) for i in wrong],      # test review i is row 5000 + i
    "label": [names[y_test[i].item()] for i in wrong],
    "model said": [names[y_pred_test[i].item()] for i in wrong]})
table

*Your answers here.*